# Install and Import Dependencies


In [1]:
!pip install tenseal syft pennylane
!pip install protobuf==3.20.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of uvicorn[standard] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.8/754.8 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.5/394.5 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 698.9/698.9 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.1/231.1 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 kB 9.9 MB/s eta 0:00:00
   ━

In [8]:
import os

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
import syft as sy
import pickle
import time
from collections import OrderedDict
from typing import List, Tuple, Dict, Optional, Callable, Union, cast
import tenseal as ts
from io import BytesIO
import numpy as np
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch
from torch import nn
import torch.nn.functional as F
import syft as sy
from logging import WARNING
import pennylane as qml
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix
import seaborn as sn
import pandas as pd
from functools import reduce

# Utility Functions


In [9]:
def choice_device(device):
    if torch.cuda.is_available() and device != "cpu":
        device = "cuda:0"
    elif (
        torch.backends.mps.is_available()
        and torch.backends.mps.is_built()
        and device != "cpu"
    ):
        device = "mps"
    else:
        device = "cpu"
    return device


def classes_string(name_dataset):
    if name_dataset == "cifar":
        return (
            "plane",
            "car",
            "bird",
            "cat",
            "deer",
            "dog",
            "frog",
            "horse",
            "ship",
            "truck",
        )
    elif name_dataset == "svhn":
        return tuple(str(i) for i in range(10))  # SVHN has 10 classes (digits 0-9)
    elif name_dataset == "caltech101":
        return tuple([f"class_{i}" for i in range(101)])  # Caltech101 has 101 classes
    elif name_dataset == "stanfordcars":
        return tuple([f"class_{i}" for i in range(196)])  # StanfordCars has 196 classes
    elif name_dataset == "fashion_mnist":
        return (
            "T-shirt/top",
            "Trouser",
            "Pullover",
            "Dress",
            "Coat",
            "Sandal",
            "Shirt",
            "Sneaker",
            "Bag",
            "Ankle boot",
        )
    else:
        raise ValueError(f"Unsupported dataset: {name_dataset}")


def save_matrix(y_true, y_pred, path, classes):
    y_true_mapped = [classes[label] for label in y_true]
    y_pred_mapped = [classes[label] for label in y_pred]
    cf_matrix_normalized = confusion_matrix(
        y_true_mapped, y_pred_mapped, labels=classes, normalize="all"
    )
    cf_matrix_round = np.round(cf_matrix_normalized, 2)
    df_cm = pd.DataFrame(
        cf_matrix_round, index=[i for i in classes], columns=[i for i in classes]
    )
    plt.figure(figsize=(12, 7))
    sn.heatmap(df_cm, annot=True)
    plt.xlabel("Predicted label", fontsize=13)
    plt.ylabel("True label", fontsize=13)
    plt.title("Confusion Matrix", fontsize=15)
    plt.savefig(path)
    plt.close()


def save_roc(targets, y_proba, path, nbr_classes):
    y_true = np.zeros(shape=(len(targets), nbr_classes))
    for i in range(len(targets)):
        y_true[i, targets[i]] = 1
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for i in range(nbr_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true[:, i], y_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    fpr["micro"], tpr["micro"], _ = roc_curve(y_true.ravel(), y_proba.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(nbr_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(nbr_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= nbr_classes
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])
    plt.figure()
    plt.plot(
        fpr["micro"],
        tpr["micro"],
        label=f"micro-average ROC curve (area = {roc_auc['micro']:.2f})",
        color="deeppink",
        linestyle=":",
        linewidth=4,
    )
    plt.plot(
        fpr["macro"],
        tpr["macro"],
        label=f"macro-average ROC curve (area = {roc_auc['macro']:.2f})",
        color="navy",
        linestyle=":",
        linewidth=4,
    )
    lw = 2
    for i in range(nbr_classes):
        plt.plot(
            fpr[i],
            tpr[i],
            lw=lw,
            label=f"ROC curve of class {i} (area = {roc_auc[i]:.2f})",
        )
    plt.plot([0, 1], [0, 1], "k--", lw=lw, label="Worst case")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Receiver operating characteristic (ROC) Curve OvR")
    plt.legend(loc="lower right")
    plt.savefig(path)
    plt.close()


def save_graphs(path_save, local_epoch, results, end_file=""):
    os.makedirs(path_save, exist_ok=True)
    print("Saving graphs in ", path_save)
    plot_graph(
        [[*range(local_epoch)]] * 2,
        [results["train_acc"], results["val_acc"]],
        "Epochs",
        "Accuracy (%)",
        ["Training accuracy", "Validation accuracy"],
        "Accuracy curves",
        path_save + "Accuracy_curves" + end_file,
    )
    plot_graph(
        [[*range(local_epoch)]] * 2,
        [results["train_loss"], results["val_loss"]],
        "Epochs",
        "Loss",
        ["Training loss", "Validation loss"],
        "Loss curves",
        path_save + "Loss_curves" + end_file,
    )


def plot_graph(
    list_xplot, list_yplot, x_label, y_label, curve_labels, title, path=None
):
    lw = 2
    plt.figure()
    for i in range(len(curve_labels)):
        plt.plot(list_xplot[i], list_yplot[i], lw=lw, label=curve_labels[i])
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    if curve_labels:
        plt.legend(loc="lower right")
    if path:
        plt.savefig(path)
    plt.close()


def get_parameters2(net, context_client=None) -> List[np.ndarray]:
    if context_client:
        encrypted_tensor = crypte(net.state_dict(), context_client)
        return [layer.get_weight() for layer in encrypted_tensor]
    return [val.cpu().numpy() for _, val in net.state_dict().items()]


def set_parameters(net, parameters: List[np.ndarray], context_client=None):
    state_dict = net.state_dict()
    params_dict = zip(state_dict.keys(), parameters)
    if context_client:
        secret_key = context_client.secret_key()
        dico = {k: deserialized_layer(k, v, context_client) for k, v in params_dict}
        new_state_dict = OrderedDict()
        for k, v in dico.items():
            if isinstance(v, CryptedLayer):
                decrypted = v.decrypt(secret_key)
                shape = state_dict[k].shape
                new_state_dict[k] = torch.Tensor(np.array(decrypted).reshape(shape))
            else:
                new_state_dict[k] = torch.Tensor(v.get_weight())
    else:
        new_state_dict = OrderedDict({k: torch.Tensor(v) for k, v in params_dict})
    net.load_state_dict(new_state_dict, strict=True)
    print("Updated model parameters")

# Security-related classes and functions


In [11]:
class Layer:
    def __init__(self, name_layer, weight):
        self.name = name_layer
        self.weight_array = weight

    def get_name(self):
        return self.name

    def get_weight(self):
        return self.weight_array

    def __add__(self, other):
        weights = other.get_weight() if isinstance(other, Layer) else other
        return Layer(self.name, self.weight_array + weights)

    def __sub__(self, other):
        weights = other.get_weight() if isinstance(other, Layer) else other
        return Layer(self.name, self.weight_array - weights)

    def __mul__(self, other):
        weights = other.get_weight() if isinstance(other, Layer) else other
        return Layer(self.name, self.weight_array * weights)

    def __truediv__(self, other):
        weights = other.get_weight() if isinstance(other, Layer) else other
        weights = self.weight_array * (1 / weights)
        return Layer(self.name, weights)

    def __len__(self):
        somme = 1
        for elem in self.weight_array.shape:
            somme *= elem
        return somme

    def shape(self):
        return self.weight_array.shape

    def sum(self, axis=0):
        return Layer(f"sum_{self.name}", self.weight_array.sum(axis=axis))

    def mean(self, axis=0):
        weights = self.weight_array.sum(axis=axis) * (1 / self.weight_array.shape[axis])
        return Layer(f"sum_{self.name}", weights)

    def decrypt(self, sk=None):
        return self.weight_array.tolist()

    def serialize(self):
        return {self.name: self.weight_array}


class CryptedLayer(Layer):
    def __init__(self, name_layer, weight, contexte=None):
        super(CryptedLayer, self).__init__(name_layer, weight)
        if isinstance(weight, (ts.tensors.CKKSTensor, bytes)):
            self.weight_array = weight
        else:
            self.weight_array = ts.ckks_tensor(contexte, weight.cpu().detach().numpy())

    def __add__(self, other):
        weights = other.get_weight() if isinstance(other, CryptedLayer) else other
        return CryptedLayer(self.name, self.weight_array + weights)

    def __sub__(self, other):
        weights = other.get_weight() if isinstance(other, CryptedLayer) else other
        return CryptedLayer(self.name, self.weight_array - weights)

    def __mul__(self, other):
        weights = other.get_weight() if isinstance(other, CryptedLayer) else other
        return CryptedLayer(self.name, self.weight_array * weights)

    def __truediv__(self, other):
        try:
            weights = other.get_weight() if isinstance(other, CryptedLayer) else other
            weights = self.weight_array * (1 / weights)
        except:
            print("Error: division operator not supported by SEAL")
            weights = []
        return CryptedLayer(self.name, weights)

    def shape(self):
        return self.weight_array.shape

    def sum(self, axis=0):
        return CryptedLayer(f"sum_{self.name}", self.weight_array.sum(axis=axis))

    def mean(self, axis=0):
        weights = self.weight_array.sum(axis=axis) * (1 / self.weight_array.shape[axis])
        return CryptedLayer(f"sum_{self.name}", weights)

    def decrypt(self, sk=None):
        return (
            self.weight_array.decrypt(sk).tolist()
            if sk
            else self.weight_array.decrypt().tolist()
        )

    def serialize(self):
        return {self.name: self.weight_array.serialize()}


def context():
    cont = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=8192,
        coeff_mod_bit_sizes=[60, 40, 40, 60],
    )
    cont.generate_galois_keys()
    cont.global_scale = 2**40
    return cont


def crypte(client_w, context_c):
    encrypted = []
    for name_layer, weight_array in client_w.items():
        if name_layer == "fc4.weight":
            encrypted.append(CryptedLayer(name_layer, weight_array, context_c))
        else:
            encrypted.append(Layer(name_layer, weight_array))
    return encrypted


def read_query(file_path):
    if os.path.exists(file_path):
        with open(file_path, "rb") as file:
            query_str = pickle.load(file)
        contexte = query_str["contexte"]
        del query_str["contexte"]
        return query_str, contexte
    else:
        print(f"File {file_path} does not exist")
        return None, None


def write_query(file_path, client_query):
    with open(file_path, "wb") as file:
        encode_str = pickle.dumps(client_query)
        file.write(encode_str)


def deserialized_layer(name_layer, weight_array, ctx):
    if isinstance(weight_array, bytes):
        return CryptedLayer(name_layer, ts.ckks_tensor_from(ctx, weight_array), ctx)
    elif isinstance(weight_array, ts.tensors.CKKSTensor):
        return CryptedLayer(name_layer, weight_array, ctx)
    else:
        return Layer(name_layer, weight_array)

In [12]:
from dataclasses import dataclass
import secrets

@dataclass
class QOTPHKeys:
    x: np.ndarray  # X mask bits, also called a/j
    z: np.ndarray  # Z mask bits, also called b/k


def qotph_keygen(n_qubits: int) -> QOTPHKeys:
    """
    Classical CSPRNG version.
    Later, you can replace this with QTRNG-generated bits for extra novelty.
    """
    x = np.array([secrets.randbits(1) for _ in range(n_qubits)], dtype=np.int64)
    z = np.array([secrets.randbits(1) for _ in range(n_qubits)], dtype=np.int64)
    return QOTPHKeys(x=x, z=z)


def qotph_apply_qotp(keys: QOTPHKeys):
    """
    Apply QOTP encryption X^a Z^b to each qubit.
    """
    for wire in range(len(keys.x)):
        if int(keys.x[wire]) == 1:
            qml.PauliX(wires=wire)
        if int(keys.z[wire]) == 1:
            qml.PauliZ(wires=wire)


def qotph_rz(theta, wire: int, keys: QOTPHKeys):
    """
    QOTPH-adjusted RZ.
    Rule: Rz(theta') where theta' = (-1)^x * theta
    """
    sign = -1.0 if int(keys.x[wire]) == 1 else 1.0
    qml.RZ(sign * theta, wires=wire)


def qotph_ry(theta, wire: int, keys: QOTPHKeys):
    """
    QOTPH-adjusted RY.
    Rule: Ry(theta') where theta' = (-1)^(x+z) * theta
    """
    parity = int(keys.x[wire]) ^ int(keys.z[wire])
    sign = -1.0 if parity == 1 else 1.0
    qml.RY(sign * theta, wires=wire)


def qotph_cnot(control: int, target: int, keys: QOTPHKeys):
    """
    Homomorphic CNOT with key update:
    x_target <- x_target XOR x_control
    z_control <- z_control XOR z_target
    """
    qml.CNOT(wires=[control, target])

    keys.x[target] ^= keys.x[control]
    keys.z[control] ^= keys.z[target]


def qotph_propagate_keys(initial_keys: QOTPHKeys, n_layers: int, n_qubits: int) -> QOTPHKeys:
    """
    Compute final keys outside the circuit so local decryption knows the final X mask.
    This must follow the exact same gate order as the QOTPH quantum circuit.
    """
    keys = QOTPHKeys(
        x=initial_keys.x.copy(),
        z=initial_keys.z.copy()
    )

    for _ in range(n_layers):
        # RZ/RY/RZ do not change keys in this simplified variational layer
        for control in range(n_qubits):
            target = (control + 1) % n_qubits
            if target != control:
                keys.x[target] ^= keys.x[control]
                keys.z[control] ^= keys.z[target]

    return keys


def qotph_decrypt_z_expectations(enc_expvals: torch.Tensor, final_keys: QOTPHKeys, device):
    """
    Local decryption for Z-basis expectation values.
    A final X mask flips the Z expectation sign.
    A final Z mask does not affect Z-basis measurement.
    """
    signs = torch.tensor(
        [(-1.0 if int(bit) == 1 else 1.0) for bit in final_keys.x],
        dtype=enc_expvals.dtype,
        device=device
    )
    return enc_expvals * signs

# ============================================================
# ALGORITHM 2: Full Encrypted-Measurement Local Decryption
# ============================================================

def qotph_decrypt_bitstring(bitstring, final_x_keys):
    """
    Algorithm 2 core: decrypt a single measured bitstring.
    
    The QOTP X^j Z^k encrypts computational basis states as:
      X^j Z^k |b⟩ = (-1)^(k·b) |b ⊕ j⟩
    
    So measuring the encrypted state gives bitstring (b ⊕ j).
    To decrypt: flip bit i wherever j_i = 1.
    Z^k only affects phase, not measurement outcome in Z-basis.
    
    Args:
        bitstring: str like "01" (measured on encrypted state)
        final_x_keys: dict {qubit_idx: j_final} after key propagation
    Returns:
        decrypted bitstring: str
    """
    bits = list(bitstring)
    for qubit_idx, j_val in final_x_keys.items():
        if j_val == 1:
            # X flips the computational basis: |b⟩ → |b⊕1⟩
            bits[qubit_idx] = '1' if bits[qubit_idx] == '0' else '0'
    return ''.join(bits)


def qotph_decrypt_histogram(encrypted_counts, final_x_keys):
    """
    Algorithm 2: decrypt an entire measurement histogram.
    
    The quantum backend returns {encrypted_bitstring: count}.
    We permute each bitstring using the X-keys to recover
    the true measurement distribution.
    
    Args:
        encrypted_counts: dict {"01": 523, "10": 501, ...}
        final_x_keys: dict {qubit_idx: j_final}
    Returns:
        decrypted_counts: dict with corrected bitstrings
    """
    decrypted_counts = {}
    for enc_bitstring, count in encrypted_counts.items():
        dec_bitstring = qotph_decrypt_bitstring(enc_bitstring, final_x_keys)
        decrypted_counts[dec_bitstring] = (
            decrypted_counts.get(dec_bitstring, 0) + count
        )
    return decrypted_counts


def histogram_to_expectations(counts, n_qubits):
    """
    Convert a measurement histogram to PauliZ expectation values.
    
    For qubit i: ⟨Z_i⟩ = P(bit_i=0) - P(bit_i=1)
    
    Args:
        counts: dict {"00": 512, "01": 488, ...}
        n_qubits: int
    Returns:
        expectations: list of floats, length n_qubits
    """
    total_shots = sum(counts.values())
    expectations = []
    for i in range(n_qubits):
        p0 = sum(c for bs, c in counts.items() if bs[i] == '0') / total_shots
        p1 = 1.0 - p0
        expectations.append(p0 - p1)
    return expectations


def qotph_algorithm2_inference(circuit_fn, features, weights, 
                                n_qubits, n_layers, shots=1024):
    """
    Full Algorithm 2 pipeline:
      1. Generate QOTP keys
      2. Build QOTPH-adjusted circuit
      3. Execute with shot-based sampling (simulating cloud)
      4. Receive encrypted histogram (cloud returns this)
      5. Decrypt histogram locally using keys
      6. Compute expectations from decrypted histogram
    
    Args:
        circuit_fn: function that builds the PennyLane circuit
        features: input feature vector (1D tensor, length 2^n_qubits)
        weights: trained PQC weights (n_layers, n_qubits, 3)
        n_qubits: int
        n_layers: int
        shots: number of measurement shots
    Returns:
        expectations: list of n_qubits floats
        metadata: dict with keys, encrypted/decrypted counts
    """
    import pennylane as qml
    
    # Step 1: Generate fresh QOTP keys
    keys = qotph_keygen(n_qubits)
    
    # Step 2: Build shot-based device and circuit
    dev_shots = qml.device("default.qubit", wires=n_qubits, shots=shots)
    
    @qml.qnode(dev_shots)
    def encrypted_circuit(inputs, w):
        # Amplitude embedding
        qml.AmplitudeEmbedding(
            features=inputs, wires=range(n_qubits),
            pad_with=0.0, normalize=True
        )
        # QOTP encryption
        for i in range(n_qubits):
            if keys[i].z: qml.PauliZ(i)
            if keys[i].x: qml.PauliX(i)
        # QOTPH-adjusted StronglyEntanglingLayers
        current_keys = {i: QOTPHKeys(x=keys[i].x, z=keys[i].z) 
                       for i in range(n_qubits)}
        for layer in range(n_layers):
            for qubit in range(n_qubits):
                rz1 = qotph_rz(float(w[layer, qubit, 0]), current_keys[qubit])
                ry  = qotph_ry(float(w[layer, qubit, 1]), current_keys[qubit])
                rz2 = qotph_rz(float(w[layer, qubit, 2]), current_keys[qubit])
                qml.RZ(rz1, wires=qubit)
                qml.RY(ry, wires=qubit)
                qml.RZ(rz2, wires=qubit)
            for qubit in range(n_qubits):
                target = (qubit + 1) % n_qubits
                qml.CNOT(wires=[qubit, target])
                current_keys = qotph_cnot(current_keys, qubit, target)
        # NO decryption — cloud returns encrypted measurements
        return qml.counts()
    
    # Step 3: Execute circuit (simulates cloud execution)
    feat_np = features.detach().cpu().numpy().flatten()
    w_np = weights.detach().cpu().numpy() if hasattr(weights, 'detach') else weights
    encrypted_counts = encrypted_circuit(feat_np, w_np)
    
    # Step 4: Propagate keys to get final state
    final_x_keys = {}
    current_keys_prop = {i: QOTPHKeys(x=keys[i].x, z=keys[i].z) 
                        for i in range(n_qubits)}
    for layer in range(n_layers):
        for qubit in range(n_qubits):
            target = (qubit + 1) % n_qubits
            current_keys_prop = qotph_cnot(current_keys_prop, qubit, target)
    for i in range(n_qubits):
        final_x_keys[i] = current_keys_prop[i].x
    
    # Step 5: CLIENT decrypts histogram locally
    decrypted_counts = qotph_decrypt_histogram(encrypted_counts, final_x_keys)
    
    # Step 6: Compute expectations from decrypted histogram
    expectations = histogram_to_expectations(decrypted_counts, n_qubits)
    
    return expectations, {
        'keys': keys,
        'encrypted_counts': encrypted_counts,
        'decrypted_counts': decrypted_counts,
        'final_x_keys': final_x_keys,
        'shots': shots,
    }

# Data setup


In [13]:
NORMALIZE_DICT = {
    "cifar": dict(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    "svhn": dict(mean=(0.4377, 0.4438, 0.4728), std=(0.1980, 0.2010, 0.1970)),
    "caltech101": dict(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    "stanfordcars": dict(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    "fashion_mnist": dict(mean=(0.2860,), std=(0.3530,)),  # Add Fashion-MNIST
}



def split_data_client(dataset, num_clients, seed):


    partition_size = len(dataset) // num_clients


    lengths = [partition_size] * (num_clients - 1)


    lengths += [len(dataset) - sum(lengths)]


    ds = random_split(dataset, lengths, torch.Generator().manual_seed(seed))


    return ds



def load_datasets(
    num_clients: int,
    batch_size: int,
    resize: int,
    seed: int,
    num_workers: int,
    splitter=10,
    dataset="cifar",
    data_path="./data/",
    data_path_val="",
):
    list_transforms = [
        transforms.ToTensor(),
        transforms.Normalize(**NORMALIZE_DICT[dataset]),
    ]

    # Resize images for non-CIFAR datasets
    if dataset in ["caltech101", "stanfordcars"] and resize is not None:
        list_transforms = [transforms.Resize((resize, resize))] + list_transforms
    elif dataset == "svhn":
        list_transforms = [
            transforms.Resize((32, 32))
        ] + list_transforms  # SVHN images are 32x32
    # No resize for Fashion-MNIST (keep 28x28)

    transformer = transforms.Compose(list_transforms)

    try:
        if dataset == "cifar":
            trainset = datasets.CIFAR10(
                data_path + dataset, train=True, download=True, transform=transformer
            )
            testset = datasets.CIFAR10(
                data_path + dataset, train=False, download=True, transform=transformer
            )
        elif dataset == "svhn":
            trainset = datasets.SVHN(
                data_path + "svhn", split="train", download=True, transform=transformer
            )
            testset = datasets.SVHN(
                data_path + "svhn", split="test", download=True, transform=transformer
            )
        elif dataset == "caltech101":
            trainset = datasets.ImageFolder(
                data_path + "caltech101/train", transform=transformer
            )
            testset = datasets.ImageFolder(
                data_path + "caltech101/test", transform=transformer
            )
        elif dataset == "stanfordcars":
            trainset = datasets.StanfordCars(
                data_path + "stanfordcars",
                split="train",
                download=True,
                transform=transformer,
            )
            testset = datasets.StanfordCars(
                data_path + "stanfordcars",
                split="test",
                download=True,
                transform=transformer,
            )
        elif dataset == "fashion_mnist":
            trainset = datasets.FashionMNIST(
                data_path + "fashion_mnist",
                train=True,
                download=True,
                transform=transformer,
            )
            testset = datasets.FashionMNIST(
                data_path + "fashion_mnist",
                train=False,
                download=True,
                transform=transformer,
            )
        else:
            raise ValueError(f"Unsupported dataset: {dataset}")
    except Exception as e:
        print(f"Failed to load dataset: {e}")
        raise

    # Split data into clients
    datasets_train = split_data_client(trainset, num_clients, seed)

    # Handle validation data
    if data_path_val:
        valset = datasets.ImageFolder(data_path_val, transform=transformer)
        datasets_val = split_data_client(valset, num_clients, seed)
    else:
        datasets_val = None

    # Create dataloaders
    trainloaders = []
    valloaders = []
    for i in range(num_clients):
        if data_path_val:
            trainloaders.append(
                DataLoader(datasets_train[i], batch_size=batch_size, shuffle=True)
            )
            valloaders.append(DataLoader(datasets_val[i], batch_size=batch_size))
        else:
            len_val = int(len(datasets_train[i]) * splitter / 100)
            len_train = len(datasets_train[i]) - len_val
            lengths = [len_train, len_val]
            ds_train, ds_val = random_split(
                datasets_train[i], lengths, torch.Generator().manual_seed(seed)
            )
            trainloaders.append(
                DataLoader(ds_train, batch_size=batch_size, shuffle=True)
            )
            valloaders.append(DataLoader(ds_val, batch_size=batch_size))

    testloader = DataLoader(testset, batch_size=batch_size)
    return trainloaders, valloaders, testloader

# Training and testing functions


In [14]:
def test(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: Union[torch.nn.Module, Tuple],
    device: torch.device,
):
    model.eval()
    test_loss, test_acc = 0, 0
    y_pred = []
    y_true = []
    y_proba = []
    softmax = nn.Softmax(dim=1)
    with torch.inference_mode():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            output = model(images)
            probas_output = softmax(output)
            y_proba.extend(probas_output.detach().cpu().numpy())
            loss = loss_fn(output, labels)
            test_loss += loss.item()
            labels = labels.data.cpu().numpy()
            y_true.extend(labels)
            preds = np.argmax(output.detach().cpu().numpy(), axis=1)
            y_pred.extend(preds)
            acc = (preds == labels).mean()
            test_acc += acc
    y_proba = np.array(y_proba)
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc * 100, y_pred, y_true, y_proba


def train_step(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: Union[torch.nn.Module, Tuple],
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> Tuple[float, float]:
    model.train()
    train_loss, train_acc = 0, 0
    for batch, (images, labels) in enumerate(dataloader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss = loss_fn(output, labels)
        train_loss += loss.item()
        loss.backward()
        optimizer.step()
        y_pred_class = torch.argmax(torch.softmax(output, dim=1), dim=1)
        train_acc += (y_pred_class == labels).sum().item() / len(output)
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc * 100


def train(
    model: torch.nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    test_dataloader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: Union[torch.nn.Module, Tuple],
    epochs: int,
    device: torch.device,
) -> Dict[str, List]:
    results = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(epochs):
        train_loss, train_acc = train_step(
            model, train_dataloader, loss_fn, optimizer, device
        )
        val_loss, val_acc, *_ = test(model, test_dataloader, loss_fn, device)
        print(
            f"\tTrain Epoch: {epoch + 1} \tTrain_loss: {train_loss:.4f} | Train_acc: {train_acc:.4f} % | "
            f"Validation_loss: {val_loss:.4f} | Validation_acc: {val_acc:.4f} %"
        )
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["val_loss"].append(val_loss)
        results["val_acc"].append(val_acc)
    return results


def serialize_ndarray(ndarray):
    if isinstance(ndarray, ts.tensors.CKKSTensor):
        return ndarray.serialize()
    elif isinstance(ndarray, torch.Tensor):
        return serialize_ndarray(ndarray.cpu().detach().numpy())
    else:
        bytes_io = BytesIO()
        np.save(bytes_io, ndarray, allow_pickle=False)
        return bytes_io.getvalue()


def deserialize_ndarray(tensor, context):
    try:
        return ts.ckks_tensor_from(context, tensor)
    except:
        bytes_io = BytesIO(tensor)
        return np.load(bytes_io, allow_pickle=False)


def serialize_parameters(parameters):
    return [serialize_ndarray(param) for param in parameters]


def deserialize_parameters(serialized_params, context):
    return [deserialize_ndarray(param, context) for param in serialized_params]


def privatize_accuracy(true_acc: float, N: int, ε=1.0):
    sensitivity = 1.0 / N
    noise = np.random.laplace(0, sensitivity / ε)
    return np.clip(true_acc + noise, 0, 1)


def accuracy_weights(accuracies: List[float], τ=0.5) -> List[float]:
    scaled_acc = [a / τ for a in accuracies]
    max_scaled = max(scaled_acc)
    exp_acc = [np.exp(a - max_scaled) for a in scaled_acc]
    sum_exp = sum(exp_acc)
    return [e / sum_exp for e in exp_acc]


def compute_difference_norm(new_param, prev_param, context):
    if isinstance(new_param, ts.tensors.CKKSTensor):
        new_dec = new_param.decrypt(context.secret_key()).tolist()
        prev_dec = prev_param.decrypt(context.secret_key()).tolist()
        diff = np.array(new_dec) - np.array(prev_dec)
    else:
        if isinstance(new_param, torch.Tensor):
            new_param = new_param.cpu().numpy()
        if isinstance(prev_param, torch.Tensor):
            prev_param = prev_param.cpu().numpy()
        diff = new_param - prev_param
    return np.linalg.norm(diff)


def aggregate_serialized(results, context, τ=0.5):
    accuracies = [dp_acc for _, dp_acc in results]
    weights = accuracy_weights(accuracies, τ)

    weights_results = [
        (deserialize_parameters(serialized_params, context), w)
        for (serialized_params, _), w in zip(results, weights)
    ]

    aggregated_params = []
    for layer_idx in range(len(weights_results[0][0])):
        layer_updates = [weights[layer_idx] for weights, _ in weights_results]
        if isinstance(layer_updates[0], ts.tensors.CKKSTensor):
            weighted_sum = sum([layer * w for layer, w in zip(layer_updates, weights)])
        else:
            weighted_sum = sum([layer * w for layer, w in zip(layer_updates, weights)])
        aggregated_params.append(weighted_sum)
    return serialize_parameters(aggregated_params)

# Main experiment setup


In [15]:
he = True
data_path = "data/"
dataset = "fashion_mnist"
yaml_path = "./results/FL/results.yml"
seed = 42
num_workers = 0
max_epochs = 10
batch_size = 32
splitter = 10
device = "gpu"
number_clients = 10
save_results = "results/FL/"
matrix_path = "confusion_matrix.png"
roc_path = "roc.png"
model_save = "fashionmnist_FedAcc.pt"
min_fit_clients = 2
min_avail_clients = 2
min_eval_clients = 2
rounds = 20
frac_fit = 1.0
frac_eval = 0.5
lr = 1e-3
path_public_key = "server_key.pkl"

DEVICE = torch.device(choice_device(device))
CLASSES = classes_string(dataset)

use_qotph = True

n_qubits = 4
n_layers = 2
weight_shapes = {"weights": (n_layers, n_qubits, 3)}  

dev = qml.device("default.qubit", wires=n_qubits)


@qml.qnode(dev, interface="torch", diff_method="backprop")
def qotph_quantum_net(inputs, weights, x_key, z_key):
    """
    QOTPH-protected quantum layer.

    The client prepares amplitude embedding, applies QOTP masking,
    then the server-side variational circuit is evaluated with
    key-adjusted gates.
    """

    qml.AmplitudeEmbedding(
        features=inputs,
        wires=range(n_qubits),
        pad_with=0.0,
        normalize=True
    )

    keys = QOTPHKeys(
        x=np.array(x_key, dtype=np.int64).copy(),
        z=np.array(z_key, dtype=np.int64).copy()
    )

    # QOTP encryption layer
    qotph_apply_qotp(keys)

    # Decomposed StronglyEntangling-style layer:
    # Rot(phi, theta, omega) = RZ(phi) RY(theta) RZ(omega)
    for layer in range(n_layers):
        for wire in range(n_qubits):
            phi = weights[layer, wire, 0]
            theta = weights[layer, wire, 1]
            omega = weights[layer, wire, 2]

            qotph_rz(phi, wire, keys)
            qotph_ry(theta, wire, keys)
            qotph_rz(omega, wire, keys)

        # CNOT ring entanglement
        for control in range(n_qubits):
            target = (control + 1) % n_qubits
            if target != control:
                qotph_cnot(control, target, keys)

    # Backend returns encrypted Z expectations
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


class QOTPHQuantumLayer(nn.Module):
    """
    Dual-mode QOTPH quantum layer:
      - Training mode: expectation-value correction (differentiable)
      - Inference mode: full Algorithm 2 (shot-based, encrypted histogram)
    """
    def __init__(self, n_qubits, n_layers, inference_shots=1024):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.inference_shots = inference_shots
        self.weights = nn.Parameter(
            0.01 * torch.randn(n_layers, n_qubits, 3)
        )
        self.use_algorithm2 = False  # Toggle for inference
        
        # Analytical device for training
        self.dev = qml.device("default.qubit", wires=n_qubits)
    
    def forward_training(self, x):
        """Current approach: differentiable expectation-value correction."""
        batch_size = x.shape[0]
        results = []
        for i in range(batch_size):
            keys = qotph_keygen(self.n_qubits)
            # ... (existing training forward code unchanged) ...
            # Returns corrected expectations via sign flip
            raw_exp = self._run_analytical_circuit(x[i], keys)
            final_keys = qotph_propagate_keys(
                keys, self.weights.detach(), self.n_layers, self.n_qubits
            )
            corrected = qotph_decrypt_z_expectations(raw_exp, final_keys)
            results.append(corrected)
        return torch.stack(results)
    
    def forward_algorithm2(self, x):
        """Algorithm 2: shot-based encrypted measurement + local decryption."""
        batch_size = x.shape[0]
        results = []
        all_metadata = []
        for i in range(batch_size):
            expectations, metadata = qotph_algorithm2_inference(
                circuit_fn=None,  # Built internally
                features=x[i],
                weights=self.weights,
                n_qubits=self.n_qubits,
                n_layers=self.n_layers,
                shots=self.inference_shots,
            )
            results.append(torch.tensor(expectations, dtype=torch.float32))
            all_metadata.append(metadata)
        self._last_metadata = all_metadata  # Store for inspection
        return torch.stack(results)
    
    def forward(self, x):
        if self.use_algorithm2 and not self.training:
            return self.forward_algorithm2(x)
        return self.forward_training(x)
    def __init__(self, n_qubits, n_layers):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.weights = nn.Parameter(
            0.01 * torch.randn(n_layers, n_qubits, 3)
        )

    def forward(self, x):
        outputs = []

        for sample in x:
            initial_keys = qotph_keygen(self.n_qubits)

            encrypted_expvals = qotph_quantum_net(
                sample,
                self.weights,
                initial_keys.x,
                initial_keys.z
            )

            if isinstance(encrypted_expvals, (list, tuple)):
                encrypted_expvals = torch.stack(list(encrypted_expvals))

            final_keys = qotph_propagate_keys(
                initial_keys,
                self.n_layers,
                self.n_qubits
            )

            decrypted_expvals = qotph_decrypt_z_expectations(
                encrypted_expvals,
                final_keys,
                device=x.device
            )

            outputs.append(decrypted_expvals)

        return torch.stack(outputs).to(dtype=x.dtype, device=x.device)

class Net(nn.Module):
    def __init__(self, num_classes=10) -> None:
        super(Net, self).__init__()

        self.network = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 2**n_qubits),
        )

        if use_qotph:
            self.qnn = QOTPHQuantumLayer(n_qubits, n_layers)
        else:
            self.qnn = qml.qnn.TorchLayer(qotph_quantum_net, weight_shapes)

        self.fc4 = nn.Linear(n_qubits, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.network(x)       # CNN feature extractor
        x = self.qnn(x)           # QOTPH / quantum layer

        # dtype fix goes AFTER qnn, BEFORE fc4
        x = x.to(dtype=self.fc4.weight.dtype, device=self.fc4.weight.device)

        x = self.fc4(x)           # final classifier logits
        return x

In [16]:
global_model = Net(num_classes=len(CLASSES)).to(DEVICE)

images, labels = next(iter(trainloaders[0]))
images = images.to(DEVICE)

with torch.no_grad():
    out = global_model(images)

print("Output shape:", out.shape)
print("Label shape:", labels.shape)
print("Output dtype:", out.dtype)
print("fc4 dtype:", global_model.fc4.weight.dtype)

NameError: name 'trainloaders' is not defined

# Main Experiment


In [ ]:
secret_path = "secret.pkl"
public_path = path_public_key
if os.path.exists(secret_path):
    with open(secret_path, "rb") as f:
        query = pickle.load(f)
    context_client = ts.context_from(query["contexte"])
else:
    context_client = context()
    with open(secret_path, "wb") as f:
        pickle.dump({"contexte": context_client.serialize(save_secret_key=True)}, f)
    with open(public_path, "wb") as f:
        pickle.dump({"contexte": context_client.serialize()}, f)
context_server = ts.context_from(read_query(public_path)[1])

trainloaders, valloaders, testloader = load_datasets(
    num_clients=number_clients,
    batch_size=batch_size,
    resize=None,
    seed=seed,
    dataset=dataset,
    data_path="./data/",
    num_workers=num_workers,
)


# Initialize global model
global_model = Net(num_classes=len(CLASSES)).to(DEVICE)
initial_params = get_parameters2(global_model, context_server)
global_serialized_params = serialize_parameters(initial_params)


# Client training function
def client_train(cid, serialized_global_params, local_epochs=max_epochs, lr=lr):
    trainloader = trainloaders[int(cid)]
    valloader = valloaders[int(cid)]
    local_model = Net(num_classes=len(CLASSES)).to(DEVICE)
    params = deserialize_parameters(serialized_global_params, context_client)
    set_parameters(local_model, params, context_client)
    optimizer = torch.optim.Adam(local_model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()
    results = train(
        local_model,
        trainloader,
        valloader,
        optimizer,
        criterion,
        epochs=local_epochs,
        device=DEVICE,
    )
    if save_results:
        save_graphs(save_results, local_epochs, results, f"_Client {cid}")
    updated_params = get_parameters2(local_model, context_client)
    serialized_updated_params = serialize_parameters(updated_params)
    num_examples = len(trainloader.dataset)
    val_acc = results["val_acc"][-1] / 100
    N_val = len(valloader.dataset)
    dp_acc = privatize_accuracy(val_acc, N_val)
    print(f"[Client {cid}] True Acc: {val_acc:.4f}, DP Acc: {dp_acc:.4f}")
    return serialized_updated_params, num_examples, dp_acc


# Federated learning simulation
print(f"Training on {DEVICE}")
start_simulation = time.time()

layer_names = list(global_model.state_dict().keys())
quantum_layer_indices = [i for i, name in enumerate(layer_names) if "qnn" in name]
previous_aggregated_params = initial_params
importance_history = [1.0] * len(initial_params)
α = 0.9
threshold = 0.001

for round_num in range(rounds):
    client_updates = []
    for cid in range(number_clients):
        print(f"[Client {cid}, round {round_num + 1}] training")
        serialized_updated_params, _, dp_acc = client_train(
            str(cid), global_serialized_params
        )
        client_updates.append((serialized_updated_params, dp_acc))

    accuracies = [dp_acc for _, dp_acc in client_updates]
    weights = accuracy_weights(accuracies)
    print(
        f"[Round {round_num + 1}] Client DP Accuracies: {accuracies}, Weights: {weights}"
    )
    global_serialized_params = aggregate_serialized(client_updates, context_server)
    new_aggregated_params = deserialize_parameters(
        global_serialized_params, context_server
    )

    differences = [
        compute_difference_norm(new, prev, context_client)
        for new, prev in zip(new_aggregated_params, previous_aggregated_params)
    ]
    importance_history = [
        α * imp + (1 - α) * diff for imp, diff in zip(importance_history, differences)
    ]
    print(f"[Round {round_num + 1}] Importance History: {importance_history}")

    frozen_layers = []
    for i, imp in enumerate(importance_history):
        if imp < threshold and i not in quantum_layer_indices:
            new_aggregated_params[i] = previous_aggregated_params[i]
            frozen_layers.append(i)
    print(f"[Round {round_num + 1}] Frozen Layers: {frozen_layers}")

    # Update global model with new aggregated parameters and evaluate
    set_parameters(global_model, new_aggregated_params, context_client)
    criterion = torch.nn.CrossEntropyLoss()
    test_loss, test_acc, _, _, _ = test(global_model, testloader, criterion, DEVICE)
    print(
        f"[Round {round_num + 1}] Global Model Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f} %"
    )

    global_serialized_params = serialize_parameters(new_aggregated_params)
    previous_aggregated_params = new_aggregated_params.copy()
    print(f"Round {round_num + 1} completed")

print(
    f"Federated learning completed. Simulation Time = {time.time() - start_simulation} seconds"
)

Training on cuda:0
[Client 0, round 1] training
Updated model parameters
	Train Epoch: 1 	Train_loss: 1.9136 | Train_acc: 37.7774 % | Validation_loss: 1.7135 | Validation_acc: 45.0658 %
	Train Epoch: 2 	Train_loss: 1.5671 | Train_acc: 51.2019 % | Validation_loss: 1.4994 | Validation_acc: 50.2193 %
	Train Epoch: 3 	Train_loss: 1.4108 | Train_acc: 53.5811 % | Validation_loss: 1.3736 | Validation_acc: 53.6184 %
	Train Epoch: 4 	Train_loss: 1.2754 | Train_acc: 55.3131 % | Validation_loss: 1.2705 | Validation_acc: 55.7566 %
	Train Epoch: 5 	Train_loss: 1.1718 | Train_acc: 56.1514 % | Validation_loss: 1.1750 | Validation_acc: 59.7039 %
	Train Epoch: 6 	Train_loss: 1.0904 | Train_acc: 62.7342 % | Validation_loss: 1.1004 | Validation_acc: 62.0066 %
	Train Epoch: 7 	Train_loss: 1.0144 | Train_acc: 64.9470 % | Validation_loss: 1.0830 | Validation_acc: 59.1557 %
	Train Epoch: 8 	Train_loss: 0.9648 | Train_acc: 65.1566 % | Validation_loss: 1.0032 | Validation_acc: 62.6645 %
	Train Epoch: 9 	Train_

DEPLOYMENT DEMO(so that QOTPH is not redundant, and helps in privacy preserving during inference)

In [ ]:
# === DEPLOYMENT SCENARIO: QOTPH protects user queries ===
# After FL training is complete, model is deployed on quantum cloud.
# New user queries are QOTPH-protected.

global_model.eval()
test_images, test_labels = next(iter(testloader))
test_image = test_images[0:1].to(DEVICE)
test_label = test_labels[0].item()

# Step 1: USER extracts CNN features locally (private)
with torch.no_grad():
    features = global_model.network(test_image)  # stays on user's device

# Step 2: USER generates QOTPH keys (private)
keys = qotph_keygen(n_qubits)

# Step 3: CLOUD runs QOTPH-adjusted PQC (sees only adjusted angles)
with torch.no_grad():
    encrypted_output = global_model.qnn(features)  # QOTPH keys used internally

# Step 4: USER runs FC4 locally (private)
with torch.no_grad():
    prediction = global_model.fc4(encrypted_output)

print(f"True: {CLASSES[test_label]}, Predicted: {CLASSES[prediction.argmax().item()]}")
print("Cloud saw: adjusted rotation angles (not user data)")
print("Cloud did NOT see: CNN features, QOTPH keys, final prediction")

In [ ]:
# === DEPLOYMENT: Full Algorithm 2 Encrypted Inference ===
print("=" * 60)
print("  DEPLOYMENT: Algorithm 2 — Encrypted Measurement + Local Decryption")
print("=" * 60)

global_model.eval()
global_model.qnn.use_algorithm2 = True  # Switch to Algorithm 2
global_model.qnn.inference_shots = 2048

test_images, test_labels = next(iter(testloader))
test_batch = test_images[:5].to(DEVICE)
true_labels = test_labels[:5]

# --- Run Algorithm 2 inference ---
with torch.no_grad():
    features = global_model.network(test_batch)
    alg2_output = global_model.qnn(features)  # Uses Algorithm 2
    predictions = global_model.fc4(alg2_output)
    pred_classes = predictions.argmax(dim=1)

# --- Also run standard (training-mode) for comparison ---
global_model.qnn.use_algorithm2 = False
with torch.no_grad():
    features = global_model.network(test_batch)
    standard_output = global_model.qnn(features)
    std_predictions = global_model.fc4(standard_output)
    std_pred_classes = std_predictions.argmax(dim=1)

# --- Display results ---
print(f"\n  {'#':<4}{'True':<14}{'Std PQC':<14}{'Alg 2':<14}{'Match'}")
print(f"  {'-'*50}")
matches = 0
for i in range(5):
    true_name = CLASSES[true_labels[i]]
    std_name = CLASSES[std_pred_classes[i]]
    alg2_name = CLASSES[pred_classes[i]]
    match = "Y" if std_pred_classes[i] == pred_classes[i] else "X"
    if match == "Y": matches += 1
    print(f"  {i:<4}{true_name:<14}{std_name:<14}{alg2_name:<14}{match}")

print(f"\n  Standard ↔ Algorithm 2 agreement: {matches}/5")

# --- Show what the cloud saw vs what the client recovered ---
print(f"\n  === What the Cloud Saw (Sample 0) ===")
meta = global_model.qnn._last_metadata[0]
print(f"  Encrypted counts: {dict(list(meta['encrypted_counts'].items())[:4])}...")
print(f"  Cloud CANNOT determine the true measurement distribution")

print(f"\n  === What the Client Recovered Locally ===")
print(f"  Final X-keys: {meta['final_x_keys']}")
print(f"  Decrypted counts: {dict(list(meta['decrypted_counts'].items())[:4])}...")
print(f"  Shots used: {meta['shots']}")

# Reset to training mode
global_model.qnn.use_algorithm2 = False

In [ ]:
# === VERIFICATION: Algorithm 2 Correctness Check ===
print("=" * 60)
print("  VERIFICATION: Algorithm 2 vs Plaintext (no encryption)")
print("=" * 60)

# Run plaintext (no QOTPH at all) vs Algorithm 2
# Build a plain quantum layer for comparison
plain_dev = qml.device("default.qubit", wires=n_qubits, shots=4096)

@qml.qnode(plain_dev)
def plain_circuit(inputs, weights):
    qml.AmplitudeEmbedding(inputs, wires=range(n_qubits), 
                           pad_with=0.0, normalize=True)
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return qml.counts()

global_model.eval()
test_img = test_images[0:1].to(DEVICE)

with torch.no_grad():
    feat = global_model.network(test_img).cpu().numpy().flatten()
    w = global_model.qnn.weights.cpu().numpy()

# Plaintext shot-based
plain_counts = plain_circuit(feat, w)
plain_exp = histogram_to_expectations(plain_counts, n_qubits)

# Algorithm 2 (encrypted + decrypted)
alg2_exp, alg2_meta = qotph_algorithm2_inference(
    None, torch.tensor(feat), 
    global_model.qnn.weights, n_qubits, n_layers, shots=4096
)

print(f"\n  Plaintext expectations:   {[f'{v:.4f}' for v in plain_exp]}")
print(f"  Algorithm 2 expectations: {[f'{v:.4f}' for v in alg2_exp]}")
diff = sum(abs(p - a) for p, a in zip(plain_exp, alg2_exp)) / n_qubits
print(f"  Mean |difference|: {diff:.4f}")
print(f"  (Difference is due to shot noise, not decryption error)")
print(f"  With infinite shots, these would be identical)")

# Show the privacy: encrypted vs decrypted distributions
print(f"\n  Encrypted histogram (cloud sees this):")
for bs, c in sorted(alg2_meta['encrypted_counts'].items()):
    print(f"    |{bs}⟩: {c} counts")
print(f"  Decrypted histogram (client recovers this):")
for bs, c in sorted(alg2_meta['decrypted_counts'].items()):
    print(f"    |{bs}⟩: {c} counts")
print(f"\n  The cloud's histogram and the true histogram are DIFFERENT")
print(f"  The cloud cannot determine which bitstrings are correct")

Validating the CKKS degradation being near zero

In [ ]:
def measure_ckks_degradation(model, testloader, device, n_cycles=20):
    """
    Repeatedly encrypt → decrypt the FC4 layer to measure 
    CKKS precision loss over multiple FL-equivalent rounds.
    """
    print(f"\n{'='*60}")
    print("  CKKS PRECISION DEGRADATION ANALYSIS")
    print(f"{'='*60}")
    
    baseline_loss, baseline_acc, _, _, _ = test(
        model, testloader, nn.CrossEntropyLoss(), device
    )
    print(f"  Baseline: acc={baseline_acc:.2f}%, loss={baseline_loss:.4f}")
    
    # Save original FC4 weights
    original_fc4 = {k: v.clone() for k, v in model.fc4.state_dict().items()}
    
    for cycle in range(1, n_cycles + 1):
        ctx = context()
        
        # Encrypt FC4
        enc_weights = {}
        for name, param in model.fc4.named_parameters():
            flat = param.detach().cpu().numpy().flatten().tolist()
            enc_weights[name] = ts.ckks_vector(ctx, flat)
        
        # Decrypt FC4 back
        state = model.fc4.state_dict()
        for name, param in model.fc4.named_parameters():
            dec = enc_weights[name].decrypt()
            state[name] = torch.tensor(dec, dtype=torch.float32
                                       ).reshape(param.shape)
        model.fc4.load_state_dict(state)
        
        if cycle % 5 == 0 or cycle == 1:
            loss, acc, _, _, _ = test(
                model, testloader, nn.CrossEntropyLoss(), device
            )
            drift = max(
                torch.abs(original_fc4[k] - model.fc4.state_dict()[k]
                         ).max().item()
                for k in original_fc4
            )
            print(f"  Cycle {cycle:>3}: acc={acc:.2f}%, "
                  f"drift={drift:.10f}, "
                  f"degradation={baseline_acc - acc:+.2f}%")
    
    return baseline_acc

# Call after FL training completes:
measure_ckks_degradation(global_model, testloader, DEVICE)

Overhead Measurement

In [ ]:
import time

model.eval()
test_batch = next(iter(testloader))[0][:16].to(DEVICE)

# Timing: Standard PQC (no QOTPH)
standard_layer = qml.qnn.TorchLayer(
    qotph_quantum_net, weight_shapes
)
standard_layer.weights = model.qnn.weights  # same weights

with torch.no_grad():
    t0 = time.time()
    for _ in range(3):
        features = model.network(test_batch)
        out_standard = standard_layer(features)
    time_standard = (time.time() - t0) / 3

# Timing: QOTPH-protected PQC
with torch.no_grad():
    t0 = time.time()
    for _ in range(3):
        features = model.network(test_batch)
        out_qotph = model.qnn(features)
    time_qotph = (time.time() - t0) / 3

# Accuracy equivalence check
diff = torch.abs(out_standard - out_qotph).max().item()

print(f"Standard PQC: {time_standard*1000:.1f} ms")
print(f"QOTPH PQC:    {time_qotph*1000:.1f} ms")
print(f"Overhead:     {time_qotph/time_standard:.2f}x")
print(f"Max output difference: {diff:.8f}")
print(f"Outputs match: {'YES' if diff < 1e-5 else 'NO'}")

# Save the final model


In [9]:
if save_results:
    os.makedirs(save_results, exist_ok=True)
    torch.save({"model_state_dict": global_model.state_dict()}, model_save)